# **Requirements to run the cloud base Yolo model**

In [ ]:
!pip install ultralytics flask flask-ngrok pyngrok
!pip install flask-cors pyngrok ultralytics
!pip install websockets nest_asyncio pyngrok ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 78.0 MB/s eta 0:00:00


In [ ]:
#FOR BEST ENGINE ONLY
!pip install tensorrt tensorrt-cu12 ultralytics pyngrok flask-cors

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
#FOR WEBCAM
import asyncio
import websockets
import cv2
import numpy as np
import json
import time
from collections import defaultdict
from ultralytics import YOLO
from pyngrok import conf, ngrok
import nest_asyncio

# Required for Google Colab to run background network loops
nest_asyncio.apply()

# NGROK SERVER SETUP (REQUIRED) REPLACE THE KEY INSIDE DEPENDING ON THE NGROL KEY
ngrok.set_auth_token("3ASqrcPQ7WGZeYryJZnuRu7zcoK_7vgmtG5DHyZfPHArEc8DX")

# Force Asia Pacific region for lower latency
conf.get_default().region = "ap"
port = 8765
public_url = ngrok.connect(port).public_url

# Convert the Ngrok HTTP URL to a secure WebSocket URL
ws_url = public_url.replace("https", "wss").replace("http", "ws")
print("=========================================================")
print("COPY THIS EXACT URL TO YOUR LAPTOP SCRIPT:")
print(ws_url)
print("=========================================================")

# PATH NG MODEL (REQUIRED)
MODEL_PATH = '/content/drive/MyDrive/Colab_Notebooks/m_model.pt'

try:
    print("Loading YOLO model...")
    model = YOLO(MODEL_PATH)
    print("Model Loaded!")
except Exception as e:
    print(f"Error loading model {e} Downloading YOLOv8n fallback.")
    model = YOLO('my_model.pt')

async def process_frame(websocket):
    print(">>> LOCAL LAPTOP CONNECTED TO WEBSOCKET")
    async for message in websocket:
        start_time = time.time()

        # Decode the raw binary string back into an image
        np_arr = np.frombuffer(message, np.uint8)
        frame = cv2.imdecode(np_arr, cv2.IMREAD_COLOR)

        if frame is None:
            continue

        # Run YOLO Inference
        results = model(frame, conf=0.5, verbose=False)

        # Extract Data
        detection_info = {'total': 0, 'labels': defaultdict(int)}
        for r in results:
            for box in r.boxes:
                c = int(box.cls)
                label = model.names[c]
                detection_info['labels'][label] += 1
                detection_info['total'] += 1

        detection_info['labels'] = dict(detection_info['labels'])

        # Calculate Speed
        detection_time = time.time() - start_time
        detection_info['inference_ms'] = round(detection_time * 1000)

        # Print the data to the Colab terminal
        print(f"Detected {detection_info['total']} fruits: {detection_info['labels']} in {detection_info['inference_ms']}ms")

        # Send the JSON data back to your local laptop
        await websocket.send(json.dumps(detection_info))

    print(">>> LOCAL LAPTOP DISCONNECTED")

async def main():
    async with websockets.serve(process_frame, "0.0.0.0", port):
        print("Starting WebSocket Server Colab")
        await asyncio.Future()

if __name__ == "__main__":
    asyncio.run(main())

COPY THIS EXACT URL TO YOUR LAPTOP SCRIPT:
wss://superprecise-lisha-unwelded.ngrok-free.dev
Loading YOLO model...
Model Loaded!
Starting WebSocket Server Colab
>>> LOCAL LAPTOP CONNECTED TO WEBSOCKET
Detected 4 fruits: {'Pear_B': 2, 'Mangosteen_A': 2} in 346ms
Detected 4 fruits: {'Pear_B': 2, 'Mangosteen_A': 2} in 34ms
Detected 4 fruits: {'Pear_B': 2, 'Mangosteen_A': 2} in 34ms
Detected 4 fruits: {'Pear_B': 2, 'Mangosteen_A': 2} in 34ms
Detected 4 fruits: {'Pear_B': 2, 'Mangosteen_A': 2} in 34ms
Detected 4 fruits: {'Pear_B': 2, 'Mangosteen_A': 2} in 34ms
Detected 4 fruits: {'Pear_B': 2, 'Mangosteen_A': 2} in 34ms
Detected 4 fruits: {'Pear_B': 2, 'Mangosteen_A': 2} in 35ms
Detected 3 fruits: {'Mangosteen_A': 1, 'Pear_B': 1, 'Mangosteen_B': 1} in 35ms
Detected 3 fruits: {'Mangosteen_A': 2, 'Pear_B': 1} in 36ms
Detected 3 fruits: {'Pear_B': 1, 'Mangosteen_A': 2} in 36ms
Detected 2 fruits: {'Pear_B': 1, 'Mangosteen_A': 1} in 36ms
Detected 3 fruits: {'Mangosteen_A': 1, 'Pear_B': 2} in 34ms


In [ ]:
#FOR MOBILE CAM
import asyncio
import websockets
import cv2
import numpy as np
import json
import time
from collections import defaultdict
from ultralytics import YOLO
from pyngrok import conf, ngrok
import nest_asyncio

# Required for Google Colab to run asynchronous loops
nest_asyncio.apply()

# NGROK SERVER SETUP
ngrok.set_auth_token("3ASqrcPQ7WGZeYryJZnuRu7zcoK_7vgmtG5DHyZfPHArEc8DX")

# Force Asia Pacific region for lower latency
conf.get_default().region = "ap"
port = 8765
public_url = ngrok.connect(port).public_url

# Convert HTTP URL to a secure WebSocket URL
ws_url = public_url.replace("https", "wss").replace("http", "ws")
print("=========================================================")
print("COPY THIS EXACT WEBSOCKET URL TO YOUR LAPTOP SCRIPT:")
print(ws_url)
print("=========================================================")

# PATH TO MODEL
MODEL_PATH = '/content/drive/MyDrive/Colab_Notebooks/best.engine'

try:
    print("Loading custom YOLO model...")
    model = YOLO(MODEL_PATH)
    print("Model Loaded!")
except Exception as e:
    print(f"Custom model error {e} Downloading YOLOv8n fallback.")
    model = YOLO('yolov8n.pt')

async def process_frame(websocket):
    async for message in websocket:
        start_time = time.time()

        # Decode the raw binary bytes back into an image array
        np_arr = np.frombuffer(message, np.uint8)
        frame = cv2.imdecode(np_arr, cv2.IMREAD_COLOR)

        if frame is None:
            continue

        # Run YOLO Inference
        results = model(frame, conf=0.5, verbose=False)

        # Extract Data
        detection_info = {'total': 0, 'labels': defaultdict(int)}
        for r in results:
            for box in r.boxes:
                c = int(box.cls)
                label = model.names[c]
                detection_info['labels'][label] += 1
                detection_info['total'] += 1

        detection_info['labels'] = dict(detection_info['labels'])

        # Calculate Speed
        detection_time = time.time() - start_time
        detection_info['inference_ms'] = round(detection_time * 1000)

        # LOGGING: See if fruits were found in Colab logs
        print(f"Detected {detection_info['labels']} in {detection_info['inference_ms']}ms")

        # Send the JSON response back through the tunnel
        await websocket.send(json.dumps(detection_info))

async def main():
    async with websockets.serve(process_frame, "0.0.0.0", port):
        print("SMARTSHELF WEBSOCKET SERVER RUNNING")
        await asyncio.Future()

if __name__ == "__main__":
    asyncio.run(main())

COPY THIS EXACT WEBSOCKET URL TO YOUR LAPTOP SCRIPT:
wss://superprecise-lisha-unwelded.ngrok-free.dev
Loading custom YOLO model...
WARNING ⚠️ Unable to automatically guess model task, assuming 'task=detect'. Explicitly define task for your model, i.e. 'task=detect', 'segment', 'classify','pose' or 'obb'.
Model Loaded!
SMARTSHELF WEBSOCKET SERVER RUNNING
Loading /content/drive/MyDrive/Colab_Notebooks/best.engine for TensorRT inference...
Detected {'Mangosteen_A': 1} in 336ms
Detected {'Mangosteen_A': 1} in 43ms
Detected {} in 40ms
Detected {} in 39ms
Detected {'Mangosteen_A': 1} in 40ms
Detected {'Mangosteen_A': 1} in 40ms
Detected {'Mangosteen_A': 1} in 39ms
Detected {'Mangosteen_A': 1} in 39ms
Detected {'Mangosteen_A': 1} in 40ms
Detected {'Mangosteen_A': 1} in 39ms
Detected {'Mangosteen_A': 1} in 41ms
Detected {'Mangosteen_A': 1} in 41ms
Detected {'Mangosteen_A': 1} in 40ms
Detected {'Mangosteen_A': 1} in 39ms
Detected {} in 39ms
Detected {'Pear_A': 1} in 40ms
Detected {'Pear_A': 1, 

KeyboardInterrupt: 

In [ ]:
#LOAD YOLO MODEL TO CROP IMAGES FOR AUTOENCODER
import torch
import torch.nn as nn
import timm
import cv2
from ultralytics import YOLO
from google.colab import drive

drive.mount('/content/drive')

# Replace the string below with the exact file path to your trained model
yolo_model = YOLO('/content/drive/MyDrive/path_to_your_model/best.pt')

class SmartShelfAutoencoder(nn.Module):
    def __init__(self):
        super().__init__()

        self.encoder = timm.create_model(
            'mobilenetv4_conv_small',
            pretrained=True,
            features_only=True
        )

        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(960, 256, kernel_size=4, stride=2, padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(32, 3, kernel_size=4, stride=2, padding=1),
            nn.Sigmoid()
        )

    def forward(self, x):
        features = self.encoder(x)
        latent_space = features[-1]

        reconstructed = self.decoder(latent_space)
        return reconstructed

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
autoencoder = SmartShelfAutoencoder().to(device)

print(f"Models successfully loaded. Running on: {device}")

In [ ]:
def process_fruit_image(image_path):
    # Run YOLO26 to find the fruit bounding box
    results = yolo_model(image_path)
    box = results[0].boxes[0].xyxy[0].cpu().numpy()
    x1, y1, x2, y2 = map(int, box)

    # Load the raw image and crop it using the YOLO coordinates
    img = cv2.imread(image_path)
    cropped_fruit = img[y1:y2, x1:x2]

    # Resize to MobileNetV4's strict 224x224 requirement
    resized_fruit = cv2.resize(cropped_fruit, (224, 224))

    # Convert to PyTorch Tensor, normalize (0 to 1 float), and add the batch dimension
    tensor_fruit = torch.tensor(resized_fruit).permute(2, 0, 1).float() / 255.0
    tensor_fruit = tensor_fruit.unsqueeze(0).to(device)

    return tensor_fruit

# --- HOW TO TEST IT ---
# fresh_tensor = process_fruit_image("path_to_your_fresh_fruit.jpg")
# output_tensor = autoencoder(fresh_tensor)